# 10 — Policy Simulation (Member 3, Q7)

Standard 90/60/30-night caps are pre-computed per neighbourhood by Member 2 (`breach_count_90/60/30`, surfaced as `listings_impacted_*` in the knowledge layer). This notebook (a) reports the standard-cap impact from the knowledge layer and (b) provides a **custom-threshold** simulator on listing-level `ttm_days_booked` for arbitrary caps (e.g. 45 nights). RESIDE (unregistered entire homes) is **Barcelona-only**.

In [1]:
import sys
from pathlib import Path
# Resolve repo root whether run from notebooks/ or repo root
_here = Path.cwd()
ROOT = _here if (_here / 'src').exists() else _here.parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, joblib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_io import PROCESSED_DIR
print('repo root:', ROOT)

repo root: /home/claude/KPMG_Airbnb_Capstone


## 1. Standard caps from the knowledge layer (per neighbourhood)

In [2]:
kl = pd.read_csv(PROCESSED_DIR / 'knowledge_layer.csv')
summary = kl.groupby('city')[['listings_impacted_90','listings_impacted_60','listings_impacted_30']].sum()
print('Entire homes impacted at each cap (sum over neighbourhoods):')
print(summary)
for city,g in kl.groupby('city'):
    print(f'\n{city} — top 5 neighbourhoods at the 90-night cap:')
    print(g.nlargest(5,'listings_impacted_90')[['geo_key','listings_impacted_90','entire_home_count','breach_rate','risk_priority_score']].to_string(index=False))

Entire homes impacted at each cap (sum over neighbourhoods):
           listings_impacted_90  listings_impacted_60  listings_impacted_30
city                                                                       
barcelona                   611                   689                   751
london                     1482                  1863                  2260

barcelona — top 5 neighbourhoods at the 90-night cap:
                        geo_key  listings_impacted_90  entire_home_count  breach_rate  risk_priority_score
         la Dreta de l'Eixample                    95                201       0.4726               100.00
             la Sagrada Família                    53                106       0.5000                88.09
                   el Poble-sec                    46                 83       0.5542                91.06
l'Antiga Esquerra de l'Eixample                    43                 94       0.4574                86.19
              la Vila de Gràcia              

## 2. Custom-threshold simulator (listing level)

For caps other than 90/60/30. Counts entire homes whose `ttm_days_booked` exceeds the cap, per neighbourhood. Runs on whatever cities' `*_listings_clean.csv` are on disk.

In [3]:
def simulate_cap(cap_nights:int)->pd.DataFrame:
    frames=[]
    for city_dir in sorted(p for p in PROCESSED_DIR.iterdir() if p.is_dir()):
        f = city_dir / f'{city_dir.name}_listings_clean.csv'
        if not f.exists():
            continue
        df = pd.read_csv(f)
        eh = df[(df['entire_home_flag']==True) & (df['ttm_days_booked']>cap_nights)]
        g = eh.groupby('subdivision').size().rename(f'impacted_{cap_nights}')
        out = g.reset_index(); out['city']=city_dir.name
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

sim45 = simulate_cap(45)
print('cities available here:', sorted(sim45['city'].unique()) if len(sim45) else 'none')
if len(sim45):
    print('\nTop 8 neighbourhoods impacted at a 45-night cap:')
    print(sim45.sort_values(sim45.columns[1], ascending=False).head(8).to_string(index=False))

cities available here: ['barcelona']

Top 8 neighbourhoods impacted at a 45-night cap:
                          subdivision  impacted_45      city
               la Dreta de l'Eixample          115 barcelona
                   la Sagrada Família           58 barcelona
                         el Poble-sec           50 barcelona
      l'Antiga Esquerra de l'Eixample           50 barcelona
                    la Vila de Gràcia           40 barcelona
Sant Pere, Santa Caterina i la Ribera           38 barcelona
                             el Raval           35 barcelona
       la Nova Esquerra de l'Eixample           34 barcelona


## 3. RESIDE proxy (Barcelona only)

Entire homes without a registration on record — potential housing units recoverable.

In [4]:
kl_bcn = kl[kl.city=='barcelona']
print('BCN unregistered entire homes (RESIDE proxy), top 8 neighbourhoods:')
print(kl_bcn.nlargest(8,'reside_unregistered_count')[['geo_key','reside_unregistered_count','entire_home_count','reside_unregistered_share']].to_string(index=False))
print('\nNOTE: RESIDE is BCN-meaningful only; London reside_unregistered_count is data context, not a regulatory breach.')

BCN unregistered entire homes (RESIDE proxy), top 8 neighbourhoods:
                              geo_key  reside_unregistered_count  entire_home_count  reside_unregistered_share
               la Dreta de l'Eixample                         59                201                     0.2935
Sant Pere, Santa Caterina i la Ribera                         39                100                     0.3900
                             el Raval                         31                 86                     0.3605
                       Gothic Quarter                         31                 71                     0.4366
                   la Sagrada Família                         30                106                     0.2830
                    la Vila de Gràcia                         20                 76                     0.2632
       la Nova Esquerra de l'Eixample                         19                 60                     0.3167
                       la Barceloneta       